In [ ]:
# ViraCut — Wan2.1-I2V-1.3B Image-to-Video
# Paramètres injectés par GitHub Actions via env vars du kernel
import os, sys, subprocess, base64, json, requests, time

PROMPT       = os.environ.get('PROMPT', 'animate this image, smooth motion')
IMAGE_B64    = os.environ.get('IMAGE_BASE64', '')
DURATION     = int(os.environ.get('DURATION', '5'))
ASPECT_RATIO = os.environ.get('ASPECT_RATIO', '9:16')
GH_TOKEN     = os.environ.get('GH_TOKEN', '')
GH_REPO      = os.environ.get('GH_REPO', '')
RUN_ID       = os.environ.get('RUN_ID', str(int(time.time())))

print(f'PROMPT: {PROMPT[:80]}')
print(f'IMAGE_B64 length: {len(IMAGE_B64)}')
print(f'DURATION: {DURATION}s | RATIO: {ASPECT_RATIO}')
print(f'GH_REPO: {GH_REPO} | RUN_ID: {RUN_ID}')

In [ ]:
# Install dependencies
subprocess.run(['pip', 'install', '-q', 'diffusers', 'transformers', 'accelerate',
                'imageio[ffmpeg]', 'opencv-python-headless', 'torch', 'torchvision',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
print('✓ Deps installées')

In [ ]:
# Décoder et sauvegarder l'image source
from PIL import Image
import io

img_bytes  = base64.b64decode(IMAGE_B64)
input_img  = Image.open(io.BytesIO(img_bytes)).convert('RGB')

# Resize selon aspect ratio
ratio_map = {'9:16': (480, 848), '16:9': (848, 480), '1:1': (624, 624)}
W, H = ratio_map.get(ASPECT_RATIO, (480, 848))
input_img = input_img.resize((W, H), Image.LANCZOS)
input_img.save('/tmp/source.jpg')
print(f'✓ Image source: {W}x{H}')

In [ ]:
# Charger Wan2.1-I2V-1.3B
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video

print('📥 Chargement Wan2.1-I2V-1.3B…')
pipe = WanImageToVideoPipeline.from_pretrained(
    'Wan-AI/Wan2.1-I2V-01-1.3B-Diffusers',
    torch_dtype=torch.float16
)
pipe = pipe.to('cuda')
pipe.enable_model_cpu_offload()
print('✓ Modèle chargé sur GPU')

In [ ]:
# Inférence
print(f'🎬 Génération vidéo — {DURATION}s @ {ASPECT_RATIO}…')

num_frames = DURATION * 8 + 1  # 8fps * durée + 1

negative_prompt = (
    'Bright tones, overexposed, static, blurred details, subtitles, '
    'style, works, paintings, images, static, overall gray, worst quality, '
    'low quality, JPEG compression residual, ugly, incomplete, extra fingers'
)

output = pipe(
    image=input_img,
    prompt=PROMPT,
    negative_prompt=negative_prompt,
    height=H,
    width=W,
    num_frames=num_frames,
    guidance_scale=5.0,
    num_inference_steps=30,
    generator=torch.Generator('cuda').manual_seed(42)
)

OUT_PATH = '/tmp/wan_output.mp4'
export_to_video(output.frames[0], OUT_PATH, fps=16)
size_mb = os.path.getsize(OUT_PATH) / 1048576
print(f'✓ Vidéo générée: {size_mb:.1f} MB → {OUT_PATH}')

In [ ]:
# Upload vers GitHub Release
if not GH_TOKEN or not GH_REPO:
    print('⚠ GH_TOKEN ou GH_REPO manquant — skip upload')
    sys.exit(0)

RELEASE_TAG  = f'viravid-{RUN_ID}'
ASSET_NAME   = f'lescrados-{RUN_ID}.mp4'
API_HEADERS  = {
    'Authorization': f'token {GH_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

# 1. Créer la Release
print(f'📦 Création GitHub Release {RELEASE_TAG}…')
rel = requests.post(
    f'https://api.github.com/repos/{GH_REPO}/releases',
    headers=API_HEADERS,
    json={
        'tag_name':   RELEASE_TAG,
        'name':       f'ViraCut Video {RUN_ID}',
        'body':       f'Prompt: {PROMPT}\nRatio: {ASPECT_RATIO} | Duration: {DURATION}s',
        'draft':      False,
        'prerelease': True
    },
    timeout=15
)

if rel.status_code not in (200, 201):
    print(f'✗ Erreur Release: {rel.status_code} — {rel.text[:200]}')
    sys.exit(1)

upload_url = rel.json()['upload_url'].replace('{?name,label}', '')
print(f'✓ Release créée: {rel.json()["html_url"]}')

# 2. Upload le MP4
print(f'📤 Upload MP4…')
with open(OUT_PATH, 'rb') as f:
    up = requests.post(
        upload_url,
        headers={**API_HEADERS, 'Content-Type': 'video/mp4'},
        params={'name': ASSET_NAME},
        data=f,
        timeout=120
    )

if up.status_code not in (200, 201):
    print(f'✗ Erreur upload: {up.status_code} — {up.text[:200]}')
    sys.exit(1)

download_url = up.json()['browser_download_url']
print(f'✓ MP4 disponible: {download_url}')
print(f'VIRAVID_URL={download_url}')  # parsé par le workflow GH Actions